In [ ]:
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Function
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    classification_report, balanced_accuracy_score,
    roc_auc_score, f1_score, confusion_matrix,
    matthews_corrcoef, cohen_kappa_score
)
import seaborn as sns
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"A usar: {device}")

In [ ]:
with open(r"/home/matildxi/Secretária/Documentos/Tese/DANN/wesad/processed_data_sequences.pkl", 'rb') as f:
    data = pickle.load(f)

X_seq      = data['X_seq']
y_seq      = data['y_seq']
groups_seq = data['groups_seq']

le = LabelEncoder()
domain_labels = le.fit_transform(groups_seq)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"A usar: {device}")

In [ ]:
class GRL(Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x

    @staticmethod
    def backward(ctx, grad):
        return grad.neg() * ctx.lam, None

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=16, nhead=4, dim_feedforward=64, 
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.proj = nn.Linear(16, 128)

    def forward(self, x):
        x = x.permute(0, 2, 1)        # (batch, 60, 16)
        x = self.transformer(x)        # (batch, 60, 16)
        x = x.mean(dim=1)              # (batch, 16) — global average
        return self.proj(x)            # (batch, 128)

In [ ]:
class StressClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.rede(x)

In [ ]:
class DomainDiscriminator(nn.Module):
    def __init__(self, n_subjects):       
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(128, 32),  
            nn.ReLU(),
            nn.Linear(32, n_subjects),  
        )

    def forward(self, x, lam):
        x = GRL.apply(x, lam)     # GRL inverte o gradiente
        return self.rede(x)

In [ ]:
class DANN(nn.Module):
    def __init__(self, n_subjects):
        super().__init__()
        self.extractor  = FeatureExtractor()
        self.stress_cls = StressClassifier()
        self.domain_dis = DomainDiscriminator(n_subjects)

    def forward(self, x, lam):
        z          = self.extractor(x)
        stress_out = self.stress_cls(z)
        domain_out = self.domain_dis(z, lam)
        return stress_out, domain_out

In [ ]:
logo     = LeaveOneGroupOut()
all_true = []
all_pred = []
fold_f1s = []

for train_idx, test_idx in logo.split(X_seq, y_seq, groups_seq):

    X_train = X_seq[train_idx]
    X_test  = X_seq[test_idx]
    y_train = y_seq[train_idx]
    y_test  = y_seq[test_idx]
    d_train = domain_labels[train_idx]

    # normalizar por canal
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std  = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
    X_train = (X_train - mean) / std
    X_test  = (X_test  - mean) / std

    # label encoding dos sujeitos
    le_fold    = LabelEncoder()
    d_train_sm = le_fold.fit_transform(d_train)
    n_subjects = len(le_fold.classes_)

    # tensores em CPU — o DataLoader trata de mover para GPU por batch
    Xt = torch.FloatTensor(X_train)
    yt = torch.LongTensor(y_train)
    dt = torch.LongTensor(d_train_sm)

    # criar modelo
    torch.cuda.empty_cache()
    model     = DANN(n_subjects=n_subjects).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn_stress = nn.CrossEntropyLoss(
        weight=torch.FloatTensor([1.0, 1.76]).to(device)
    )
    loss_fn_domain = nn.CrossEntropyLoss()

    # DataLoader com mini-batches
    dataset    = TensorDataset(Xt, yt, dt)
    dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

    # treino
    model.train()
    for epoch in range(250):
        lam = 2 / (1 + np.exp(-10 * epoch / 250)) - 1

        for Xb, yb, db in dataloader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            db = db.to(device)

            stress_out, domain_out = model(Xb, lam)

            loss_stress = loss_fn_stress(stress_out, yb)
            loss_domain = loss_fn_domain(domain_out, db)
            loss_total  = loss_stress + 0.1 * loss_domain

            optimizer.zero_grad()
            loss_total.backward()
            optimizer.step()

    # avaliação em batches
    model.eval()
    test_ds = TensorDataset(torch.FloatTensor(X_test))
    test_dl = DataLoader(test_ds, batch_size=256)

    all_preds = []
    with torch.no_grad():
        for (Xb,) in test_dl:
            out, _ = model(Xb.to(device), lam=0)
            all_preds.append(out.argmax(dim=1).cpu())

    y_pred = torch.cat(all_preds).numpy()

    all_true.extend(y_test)
    all_pred.extend(y_pred)
    fold_f1s.append(f1_score(y_test, y_pred, average='macro', zero_division=0))

In [ ]:
# converter para arrays
all_true = np.array(all_true)
all_pred = np.array(all_pred)

print("=" * 50)
print("Estatísticas FINAIS - DANN + IG + SMOTE")
print("=" * 50)

print(classification_report(all_true, all_pred,
                             target_names=['baseline', 'stress'],
                             zero_division=0))

print(f"Balanced Accuracy:        {balanced_accuracy_score(all_true, all_pred):.3f}")
print(f"Macro F1 médio por fold:  {np.mean(fold_f1s):.3f}")
print(f"Desvio Padrão F1 (LOSO):  {np.std(fold_f1s):.3f}")
print(f"Matthews Corr. Coef.:     {matthews_corrcoef(all_true, all_pred):.3f}")
print(f"Cohen's Kappa:            {cohen_kappa_score(all_true, all_pred):.3f}")

# F1 por fold (para ver estabilidade entre sujeitos)
print(f"\nF1 por fold:")
for i, f1 in enumerate(fold_f1s):
    print(f"  Sujeito {i+1:02d}: {f1:.3f}")

# matriz de confusão
plt.figure(figsize=(7, 5))
sns.heatmap(confusion_matrix(all_true, all_pred),
            annot=True, fmt='d', cmap='Blues',
            xticklabels=['baseline', 'stress'],
            yticklabels=['baseline', 'stress'])
plt.title('Confusion Matrix — DANN + IG + SMOTE')
plt.ylabel('Real'); plt.xlabel('Predito')
plt.tight_layout()
plt.show()

# F1 por fold em gráfico — mostra estabilidade inter-sujeito
plt.figure(figsize=(10, 4))
plt.bar(range(1, len(fold_f1s)+1), fold_f1s, color='steelblue')
plt.axhline(np.mean(fold_f1s), color='red', linestyle='--',
            label=f'Média: {np.mean(fold_f1s):.3f}')
plt.xlabel('Sujeito (fold LOSO)')
plt.ylabel('Macro F1')
plt.title('F1 por Sujeito — DANN + IG + SMOTE')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# F1 @ Youden-optimal threshold
from sklearn.metrics import roc_curve, f1_score
import numpy as np
_yt = np.array(all_true); _ys = np.array(y_score)
_fpr, _tpr, _thr = roc_curve(_yt, _ys)
_t_opt = float(_thr[np.argmax(_tpr - _fpr)])
_f1opt = f1_score(_yt, (_ys >= _t_opt).astype(int), average='macro', zero_division=0)
print(f'F1@opt : {_f1opt:.3f}  (threshold={_t_opt:.3f})')